# `pt.pivot` / `df.pivot` — Excel-Style 2D Pivot Tables

`pt.pivot()` summarizes and aggregates tabular data across dimensions using the intuitive **RCVA** grammar:
- **`r`** (Rows): Row grouping dimensions (*optional*)
- **`c`** (Cols): Column header dimensions (*optional*)
- **`v`** (Values): Value / measure column(s) to aggregate (**required**)
- **`a`** (Aggregation): Aggregation function (`sum`, `mean`, `median`, `min`, `max`, `count`, `std`, `n`; default: `sum`)

### Core Guarantees:
1. **Always Flat 1D Output**: Never produces MultiIndex rows or tuples in column headers. Column names are joined with `_` (e.g. `Sales_2023`).
2. **Always Clean Reset Index**: Returns a standard `RangeIndex(0, 1, 2, ...)` every time.
3. **Automatic Subsetting**: Only references `r`, `c`, and `v`—no need to pre-select columns.
4. **No Silent Data Loss**: `dropna=False` by default, retaining missing categories.

In [1]:
import pytae as pt
import pandas as pd

penguins = pt.sample("penguins")
flights = pt.sample("flights")
penguins.head()

species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
str,str,f64,f64,f64,f64,str
"""Adelie""","""Torgersen""",39.1,18.7,181.0,3750.0,"""Male"""
"""Adelie""","""Torgersen""",39.5,17.4,186.0,3800.0,"""Female"""
"""Adelie""","""Torgersen""",40.3,18.0,195.0,3250.0,"""Female"""
"""Adelie""","""Torgersen""",null,null,null,null,null
"""Adelie""","""Torgersen""",36.7,19.3,193.0,3450.0,"""Female"""


## 1. Classic 2D Contingency Grid (`r=`, `c=`, `v=`)

Summarize average body mass across islands (rows) and species (columns):

In [2]:
penguins.pivot(r="island", c="species", v="body_mass_g", a="mean")

island,Adelie,Chinstrap,Gentoo
str,f64,f64,f64
"""Biscoe""",3709.659091,null,5076.01626
"""Dream""",3688.392857,3733.088235,null
"""Torgersen""",3706.372549,null,null


## 2. Frequency Matrix via Row Counts ()

Count occurrences across dimensions (the direct successor to cross-tabulation):


In [3]:
penguins.pivot(r="island", c="species", a="count")

island,Adelie,Chinstrap,Gentoo
str,i64,i64,i64
"""Biscoe""",44,0,124
"""Dream""",56,68,0
"""Torgersen""",52,0,0


## 3. Flexible Excel Layouts: Optional Rows or Columns

Just like dragging fields in Excel, `r` and `c` are completely optional.

In [4]:
# Omit c: vertical summary table by row group
penguins.pivot(r="species", v="body_mass_g", a="mean")

species,body_mass_g
str,f64
"""Adelie""",3700.662252
"""Chinstrap""",3733.088235
"""Gentoo""",5076.01626


In [5]:
# Omit r: horizontal summary table across column headers
penguins.pivot(c="species", v="body_mass_g", a="mean")

Adelie,Chinstrap,Gentoo
f64,f64,f64
3700.662252,3733.088235,5076.01626


In [6]:
# Omit both r and c: grand scalar aggregation across the dataset
penguins.pivot(v="body_mass_g", a="mean")

body_mass_g
f64
4201.754386


## 4. Multi-Level Dimensions & Multi-Metric Reporting

Pass a list of columns to `r`, `c`, or `v`:

In [7]:
# Hierarchical rows + multiple metric values
penguins.pivot(
    r=["island", "sex"],
    c="species",
    v=["body_mass_g", "flipper_length_mm"],
    a="mean",
    f_v=0
)


island,sex,body_mass_g_Adelie,body_mass_g_Chinstrap,body_mass_g_Gentoo,flipper_length_mm_Adelie,flipper_length_mm_Chinstrap,flipper_length_mm_Gentoo
str,str,f64,f64,f64,f64,f64,f64
"""Biscoe""",null,null,null,null,0.0,0.0,215.75
"""Biscoe""",null,0.0,0.0,4587.5,null,null,null
"""Biscoe""","""Female""",3369.318182,0.0,4679.741379,187.181818,0.0,212.706897
"""Biscoe""","""Male""",4050.0,0.0,5484.836066,190.409091,0.0,221.540984
"""Dream""",null,null,null,null,179.0,0.0,0.0
…,…,…,…,…,…,…,…
"""Dream""","""Male""",4045.535714,3938.970588,0.0,191.928571,199.911765,0.0
"""Torgersen""",null,null,null,null,187.25,0.0,0.0
"""Torgersen""",null,3681.25,0.0,0.0,null,null,null


## 5. Seamless Method Chaining & Plotting

Because `pt.pivot` always returns a clean, rectangular DataFrame with a flat RangeIndex, you can immediately chain into further transformations or `.plot()`:

In [8]:
piv = penguins.pivot(r="island", c="species", v="body_mass_g", a="mean", f_v=0)
# Clean flat columns and standard index ready for downstream manipulation:
print("Columns:", list(piv.columns))
print("Index:", list(piv.index))
piv

Columns: ['island', 'Adelie', 'Chinstrap', 'Gentoo']
Index: [0, 1, 2]


island,Adelie,Chinstrap,Gentoo
str,f64,f64,f64
"""Biscoe""",3709.659091,0.0,5076.01626
"""Dream""",3688.392857,3733.088235,0.0
"""Torgersen""",3706.372549,0.0,0.0
